# Destilar la voz **Alex** (Kokoro) a Piper — 2 celdas
Timbre de `em_alex`, sin grabar nada. Entrenamiento con **piper1-gpl** (comandos copiados de un notebook comunitario probado).

**Antes:** Entorno de ejecución → Cambiar tipo → **T4 GPU**.
**Celda 1** (~15-20 min) · **Celda 2** (2-4 h; se puede cortar y reejecutar para exportar).

In [ ]:
#@title 1. PREPARAR TODO (Kokoro genera el dataset + instala Piper) — ~15-20 min
import torch, os
assert torch.cuda.is_available(), "Activá T4: Entorno de ejecución -> Cambiar tipo de entorno -> GPU"
print("GPU:", torch.cuda.get_device_name(0))
!pip install -q kokoro-onnx==0.5.0
!wget -q -nc -O /content/kokoro.onnx "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx"
!wget -q -nc -O /content/voices.bin "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin"
!wget -q -O /content/corpus.txt "https://raw.githubusercontent.com/Lazy-Money/Loud-Web/claude/readvox-research-slumjx/colab/corpus/corpus_es_1300.txt"
import wave, numpy as np
from kokoro_onnx import Kokoro
frases = [l.strip() for l in open("/content/corpus.txt", encoding="utf-8") if l.strip()]
print(f"{len(frases)} frases. Generando audio con la voz Alex...")
kokoro = Kokoro("/content/kokoro.onnx", "/content/voices.bin")
os.makedirs("/content/dataset/wavs", exist_ok=True)
rows, total = [], 0.0
for i, f in enumerate(frases):
    try: s, r = kokoro.create(f, voice="em_alex", speed=1.0, lang="es")
    except Exception: continue
    idx = np.linspace(0, len(s)-1, int(len(s)*22050/r))
    d = np.clip(np.interp(idx, np.arange(len(s)), s)*32767, -32768, 32767).astype(np.int16)
    if not 1.0 <= len(d)/22050 <= 20.0: continue
    n = f"f{i:05d}.wav"
    with wave.open(f"/content/dataset/wavs/{n}","wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(22050); w.writeframes(d.tobytes())
    rows.append(f"{n}|{f}"); total += len(d)/22050
    if len(rows)%200==0: print(f"  {len(rows)} frases...")
open("/content/dataset/metadata.csv","w",encoding="utf-8").write("\n".join(rows)+"\n")
print(f"Dataset: {len(rows)} clips, {total/60:.1f} min.")

print("Instalando Piper (piper1-gpl, receta probada)...")
!apt-get -q update -y > /dev/null 2>&1
!apt-get -q install -y build-essential cmake ninja-build espeak-ng > /dev/null 2>&1
%cd /content
!git clone -q https://github.com/OHF-voice/piper1-gpl.git
%cd /content/piper1-gpl
!pip install -q -e '.[train]'
!pip install -q --upgrade scikit-build "protobuf==3.20.3"
!bash build_monotonic_align.sh > /dev/null 2>&1
!python setup.py build_ext --inplace > /dev/null 2>&1
!wget -q -nc -O /content/base.ckpt "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/es/es_ES/davefx/medium/epoch%3D2218-step%3D562840.ckpt"
print("\n=== LISTO. Ahora corré la celda 2 para entrenar. ===")

In [ ]:
#@title 2. ENTRENAR Y DESCARGAR — 2-4 h (podés cortarla y reejecutarla para exportar lo entrenado)
import glob, os
def last_ckpt():
    c = sorted(glob.glob("/content/train_out/**/*.ckpt", recursive=True), key=os.path.getmtime)
    return c[-1] if c else None
if not last_ckpt():
    !cd /content/piper1-gpl && python -m piper.train fit \
      --data.voice_name "es_alex_kokoro" \
      --data.csv_path /content/dataset/metadata.csv \
      --data.audio_dir /content/dataset/wavs \
      --data.espeak_voice es \
      --data.cache_dir /content/cache \
      --data.config_path /content/es_alex_kokoro.onnx.json \
      --data.batch_size 16 \
      --model.sample_rate 22050 \
      --data.validation_split 0 --data.num_test_examples 0 \
      --trainer.default_root_dir /content/train_out \
      --trainer.accelerator gpu --trainer.devices 1 \
      --trainer.max_epochs 1000 \
      --trainer.precision 16-mixed \
      --checkpoint.every_n_epochs 25 --checkpoint.save_top_k -1 --checkpoint.monitor null \
      --ckpt_path /content/base.ckpt
ck = last_ckpt()
assert ck, "No hay checkpoint todavía: dejá entrenar unos minutos (se guarda cada 25 epochs) y reejecutá esta celda."
!cd /content/piper1-gpl && python -m piper.train.export_onnx --checkpoint "{ck}" --output-file /content/es_alex_kokoro.onnx
from google.colab import files
files.download("/content/es_alex_kokoro.onnx")
files.download("/content/es_alex_kokoro.onnx.json")
print("Copiá ambos a tu carpeta de voces de LoudVox y elegí 'es_alex_kokoro' en Configuración")